In [ ]:
pip install langchain-ollama

In [ ]:
pip install scikit-learn

In [5]:
#Chat models
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="qwen3:1.7b",
    temperature=0,
)
response = llm.invoke("Who are you")
print(response.content)

I am an AI assistant developed by Alibaba Cloud, designed to help users with various tasks. My main functions include providing information, answering questions, assisting with tasks, and offering creative ideas. I aim to be helpful, polite, and accurate in my responses. 

I cannot perform tasks that require physical interaction or access to sensitive data. If you have any questions or need assistance, feel free to ask! 😊


## Document Similarity using Embeddings

In [60]:
#Embedding Models
from langchain_ollama import OllamaEmbeddings
embedding_model= OllamaEmbeddings(model="qwen3-embedding:8b")

#Doc similarity
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
docs=["The sun rises in the east.", 
      "Water boils at 100°C at sea level.", 
      "Einstein revolutionized physics with relativity.", 
      "A cat sleeps for 15 hours a day.", 
      "Mountains are formed by tectonic activity."]
query="what are mountains"

doc_embeddings= embedding_model.embed_documents(docs)
query_embedding= embedding_model.embed_query(query)
scores = cosine_similarity([query_embedding],doc_embeddings)[0]
index,score= sorted(list(enumerate(scores)),key=lambda x:x[1])[-1]
print(f"Similar doc to:  {query}: {docs[index]} with similarity score of {score}")


Similar doc to:  what are mountains: Mountains are formed by tectonic activity. with similarity score of 0.6433861808315138


### Prompt template

In [67]:
#Prompts
from langchain_core.prompts import PromptTemplate
template = PromptTemplate(
    template = """
    Please tell 5 lines about {input_topic} in {input_language}
    """,
    input_variables=['input_topic','input_language']
)
topic="Dance"
language="Hindi"
prompt= template.invoke({
    'input_topic': topic,
    'input_language': language
}
)
response = llm.invoke(prompt)
print(response.content)


नृत्य हिंदी में एक अद्भुत कला है, जो रंग, रूप और भाव को जीवंत करता है।  
हाथों की भावना और पैरों की धुन से यह दुनिया के रहस्यों को बयान करता है।  
भारतीय नृत्य विश्व की संस्कृति का एक अमूल्य अंग है, जो प्राचीनता से जुड़ा है।  
इसमें शारीरिक गति और मन की धुन का संगम होता है, जो दर्शक को आकर्षित करता है।  
हर नृत्य एक कहानी कहता है, जो हृदय को छूता है और जीवन की खुशबू बनाता है।


### Messages

In [69]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain_ollama import ChatOllama
model = ChatOllama(
    model="qwen3-vl:8b",
    temperature=0,
)
messages=[
    SystemMessage(content="You are a medical ai assistance"),
    HumanMessage(content="why paracetamol is least dangeorours OTC medicine.")
         ]
response= model.invoke(messages)
messages.append(AIMessage(content=response.content))
print(messages)


[SystemMessage(content='You are a medical ai assistance', additional_kwargs={}, response_metadata={}), HumanMessage(content='why paracetamol is least dangeorours OTC medicine.', additional_kwargs={}, response_metadata={}), AIMessage(content='That\'s a **common misconception**, and it\'s important to clarify: **Paracetamol (acetaminophen) is *not* inherently the "least dangerous" OTC medicine.** While it\'s **generally safe when used correctly**, it carries **significant risks if misused**, and it\'s **not universally safer than other common OTC drugs** like ibuprofen or aspirin. The idea that it\'s "least dangerous" often stems from **misunderstanding its risks or comparing it to other medications in specific contexts.**\n\nHere\'s a breakdown of why this misconception exists, and the **critical realities**:\n\n### Why People Think It\'s "Least Dangerous" (The Misconception)\n1.  **Widespread Use & Familiarity:** It\'s one of the most common OTC pain relievers globally. People often as

## History

In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

chat_template= ChatPromptTemplate([
    ('system','You are a helpful customer support agent'),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human','{query}')   
])

chat_history=[]
with open('chat_history.txt') as f:
    chat_history.extend(f.readlines())
chat_template.invoke({'chat_history':chat_history,'query':"Where is my refund"})

In [ ]:
from typing import TypedDict, Annotated, Optional, Literal

from langchain_ollama import ChatOllama
model = ChatOllama(
    model="qwen3-vl:8b",
    temperature=0,
)

# schema
class Review(TypedDict):

    key_themes: Annotated[list[str], "Write down all the key themes discussed in the review in a list"]
    summary: Annotated[str, "A brief summary of the review"]
    sentiment: Annotated[Literal["pos", "neg"], "Return sentiment of the review either negative, positive or neutral"]
    pros: Annotated[Optional[list[str]], "Write down all the pros inside a list"]
    cons: Annotated[Optional[list[str]], "Write down all the cons inside a list"]
    name: Annotated[Optional[str], "Write the name of the reviewer"]
    

structured_model = model.with_structured_output(Review)

result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Sandeep Rana
""")

print(result['name'])